In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt488\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
df = pd.read_csv("Sentiment_Analysis_Dataset.csv")
print(df.head())
print(df.shape)
print(df.columns)

                                              review sentiment
0   I absolutely loved this movie, it was fantastic!  positive
1  The product quality is excellent and worth the...  positive
2  Amazing experience, I would definitely recomme...  positive
3   The service was fast, friendly and very helpful.  positive
4  This restaurant has delicious food and great s...  positive
(40, 2)
Index(['review', 'sentiment'], dtype='object')


In [12]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def clean_text(text):
    text = str(text).lower()
    
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words
    ]
    
    return ' '.join(words)

df['cleaned_review'] = df['review'].apply(clean_text)

df[['review', 'cleaned_review']].head()

,review,cleaned_review
0,"I absolutely loved this movie, it was fantastic!",absolut love movi fantast
1,The product quality is excellent and worth the...,product qualiti excel worth money
2,"Amazing experience, I would definitely recomme...",amaz experi would definit recommend
3,"The service was fast, friendly and very helpful.",servic fast friendli help
4,This restaurant has delicious food and great s...,restaur delici food great staff


In [15]:
X = df['cleaned_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (32, 169)
Testing shape: (8, 169)


In [17]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred, pos_label='positive')
lr_recall = recall_score(y_test, lr_pred, pos_label='positive')
lr_f1 = f1_score(y_test, lr_pred, pos_label='positive')

print("Logistic Regression")
print("Accuracy :", lr_accuracy)
print("Precision:", lr_precision)
print("Recall   :", lr_recall)
print("F1 Score :", lr_f1)

Logistic Regression
Accuracy : 0.375
Precision: 0.4
Recall   : 0.5
F1 Score : 0.4444444444444444


In [18]:
nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

nb_accuracy = accuracy_score(y_test, nb_pred)
nb_precision = precision_score(y_test, nb_pred, pos_label='positive')
nb_recall = recall_score(y_test, nb_pred, pos_label='positive')
nb_f1 = f1_score(y_test, nb_pred, pos_label='positive')

print("Naive Bayes")
print("Accuracy :", nb_accuracy)
print("Precision:", nb_precision)
print("Recall   :", nb_recall)
print("F1 Score :", nb_f1)

Naive Bayes
Accuracy : 0.25
Precision: 0.25
Recall   : 0.25
F1 Score : 0.25


In [19]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes'],
    'Accuracy': [lr_accuracy, nb_accuracy],
    'Precision': [lr_precision, nb_precision],
    'Recall': [lr_recall, nb_recall],
    'F1 Score': [lr_f1, nb_f1]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.375,0.40,0.50,0.444444
1,Naive Bayes,0.250,0.25,0.25,0.250000


In [20]:
print("Logistic Regression Report")
print(classification_report(y_test, lr_pred))

print("\nNaive Bayes Report")
print(classification_report(y_test, nb_pred))

Logistic Regression Report
              precision    recall  f1-score   support

    negative       0.33      0.25      0.29         4
    positive       0.40      0.50      0.44         4

    accuracy                           0.38         8
   macro avg       0.37      0.38      0.37         8
weighted avg       0.37      0.38      0.37         8


Naive Bayes Report
              precision    recall  f1-score   support

    negative       0.25      0.25      0.25         4
    positive       0.25      0.25      0.25         4

    accuracy                           0.25         8
   macro avg       0.25      0.25      0.25         8
weighted avg       0.25      0.25      0.25         8



In [22]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    
    review_tfidf = tfidf.transform([cleaned])
    
    lr_result = lr_model.predict(review_tfidf)[0]
    nb_result = nb_model.predict(review_tfidf)[0]
    
    print("Review:", review)
    print("Logistic Regression:", lr_result)
    print("Naive Bayes:", nb_result)

In [ ]:
predict_sentiment(
    "This movie was amazing and I really enjoyed it"
)

In [ ]:
predict_sentiment(
    "The product was terrible and completely useless"
)

In [ ]:
predict_sentiment(
    "I absolutely love this product"
)